In [2]:
from merge_tables.db.connection import connect_to_postgres_via_duckdb

In [4]:
duck = connect_to_postgres_via_duckdb()

✓ Successfully connected DuckDB to PostgreSQL database 'medisoft'


In [43]:
def match_samedi_medisoft(city):
    duck.sql(
        fr"""
        with matches as (
            select 
                s.*,
                m.rec_id as besch_id,
                case
                    when strptime(regexp_extract(m.identifikation, '(\d{{2}}\.\d{{2}}\.\d{{4}})'), '%d.%m.%Y')::date = s.Geburtsdatum::date then 1
                    else 0
                end as birthdate_match,
                f.pfad as firm_name 
            from read_csv('/Users/adrienblanquer/Downloads/samedi_{city}.csv') as s
            left join pg.medisoft.table_beschaeftigte as m 
                on lower(familienname) = lower(s."Name") 
                and lower(m.vorname) = lower(s."Vorname")
            left join pg.medisoft.table_firmenstruktur as f
                on m.abetrieb_id = f.rec_id
            order by zeile
        )
        select * from matches
        qualify row_number() over (partition by zeile order by birthdate_match desc) = 1
        order by zeile
        """
    ).to_csv(f"{city}_samedi_x_medisoft.csv")

In [ ]:
duck.sql(
    r"""
    with matches as (
        select 
            s.*,
            m.rec_id as besch_id,
            case
                when strptime(regexp_extract(m.identifikation, '(\d{2}\.\d{2}\.\d{4})'), '%d.%m.%Y')::date = s.Geburtsdatum::date then 1
                else 0
            end as birthdate_match,
            f.pfad as firm_name 
        from read_csv('/Users/adrienblanquer/Downloads/samedi_koln.csv') as s
        left join pg.medisoft.table_beschaeftigte as m 
            on lower(familienname) = lower(s."Name") 
            and lower(m.vorname) = lower(s."Vorname")
        left join pg.medisoft.table_firmenstruktur as f
            on m.abetrieb_id = f.rec_id
        order by zeile
    )
    select * from matches
    qualify row_number() over (partition by zeile order by birthdate_match desc) = 1
    order by zeile
    """
).to_csv("koln_samedi_x_medisoft.csv")

In [44]:
match_samedi_medisoft('dusseldorf')

In [41]:
duck.sql(
    """
    select * from pg.medisoft.table_beschaeftigte where rec_id in ('00_A1I00JKRKH')
    """
)

┌───────────────┬─────────────────────────┬──────────────┬─────────┬───────────────┬─────────┬────────────┬─────────────────────┬────────────┬─────────┬────────────┬───────────────┬───────────────┬──────────────────────────────────────────────────────────────────────────────────┬───────────────┬────────────────┬──────────────────────────────────────────┬───────────────────┬───────────────┬────────────┬──────────────────────────┬───────────────────────┬─────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────┬─────────┬─────────────┬────────────┬────────────────┬────────────┬──────────────┬─────────────┬─────────────────┬───────────────────────────────────────────────────────────────────────────────────────────

In [42]:
duck.sql(
    """
    select * from pg.medisoft.table_beschaeftigte where familienname like 'arba%'
    """
)

┌───────────────┬─────────────────────────────────┬──────────────┬──────────────┬─────────────────────┬─────────┬─────────┬─────────────────────┬────────────┬─────────┬────────────┬──────────────────────────────────────┬──────────────────────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────┬───────────────┬────────────────┬──────────────────────────────────────────┬───────────────────┬───────────────┬────────────┬──────────────────────────┬───────────────────────┬─────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────┬─────────────┬─────────────┬────────────┬────────────────┬────────────┬──────────────┬─────────────┬─────────────────┬──────────┬─────────────┬─────────────┬─────────────┬────────────────

In [12]:
duck.sql(
    """
    select 
        s.*,
        m.rec_id as besch_id,
        case
            when strptime(regexp_extract(m.identifikation, '(\d{2}\.\d{2}\.\d{4})'), '%d.%m.%Y')::date = s.Geburtsdatum::date then 1
            else 0
        end as birthdate_match,
        m.telefon,
        f.pfad as firm_name,
        p.phone,
        p.mobile
    from read_csv('/Users/adrienblanquer/Downloads/samedi_koln.csv') as s
    left join read_csv('/Users/adrienblanquer/Downloads/patients_2026-03-10_15-51-44.csv') as p
        on lower(s."Name") = lower(p.last_name)
        and lower(s."Vorname") = lower(p.first_name)
    left join pg.medisoft.table_beschaeftigte as m 
        on (lower(familienname) = lower(s."Name") 
        and lower(m.vorname) = lower(s."Vorname"))
        or coalesce(p.phone, p.mobile) = m.telefon
    left join pg.medisoft.table_firmenstruktur as f
        on m.abetrieb_id = f.rec_id
    where s."Name" is not null and p.city = 'Köln' and m.telefon is not null
    order by zeile

    """
)

<>:7: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:7: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
/var/folders/r3/svzsdlzn73xd8g34qr5rhw9h0000gq/T/ipykernel_56967/3503300080.py:7: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
  when strptime(regexp_extract(m.identifikation, '(\d{2}\.\d{2}\.\d{4})'), '%d.%m.%Y')::date = s.Geburtsdatum::date then 1


┌───────┬────────────┬──────────┬─────────────┬────────────────────────────────────────────────┬─────────┬─────────┬─────────┬─────────┬──────────────┬──────────────┬─────────────┬──────────────────┬───────────┬───────────────┬─────────────────┬───────────────────┬────────────────────────────────────────────────────────┬─────────┬────────────────┐
│ Zeile │   Datum    │ Uhrzeit  │  Ressource  │                     Termin                     │  Dauer  │ Pat-Nr. │  Name   │ Vorname │ Versicherung │ Geburtsdatum │ Gebucht von │    Gebucht am    │ Kommentar │   besch_id    │ birthdate_match │      telefon      │                       firm_name                        │  phone  │     mobile     │
│ int64 │    date    │   time   │   varchar   │                    varchar                     │ varchar │ varchar │ varchar │ varchar │   varchar    │     date     │   varchar   │     varchar      │  varchar  │    varchar    │      int32      │      varchar      │                        varchar    

In [7]:
duck.sql(
    """
    select * from read_csv('/Users/adrienblanquer/Downloads/patients_2026-03-10_15-51-44.csv') where last_name = 'Hilger'
    """
)

┌─────────┬───────────┬────────────┬─────────┬─────────┬─────────┬─────────┬───────────────────────┐
│  city   │ last_name │ first_name │  phone  │ mobile  │  email  │ address │       birthdate       │
│ varchar │  varchar  │  varchar   │ varchar │ varchar │ varchar │ varchar │        varchar        │
├─────────┼───────────┼────────────┼─────────┼─────────┼─────────┼─────────┼───────────────────────┤
│ Köln    │ Hilger    │ Manuela    │ NULL    │ NULL    │ NULL    │ ., . .  │ 18.03.1977 (48 Jahre) │
└─────────┴───────────┴────────────┴─────────┴─────────┴─────────┴─────────┴───────────────────────┘

# 2026 full samedi plannings

In [ ]:
duck.sql(
    """
    select 
        *,
        case 
            when 'Kommentar' in (select column_name from information_schema.columns where table_name = '/Users/adrienblanquer/Downloads/samedi_koln.csv') 
            then Kommentar
            else NULL
        end as Kommentar,
        'koln' as standort
    from read_csv('/Users/adrienblanquer/Downloads/samedi_koln.csv')

    union all

    select 
        *,
        case 
            when 'Kommentar' in (select column_name from information_schema.columns where table_name = '/Users/adrienblanquer/Downloads/samedi_dusseldorf.csv') 
            then Kommentar
            else NULL
        end as Kommentar,
        'dusseldorf' as standort
    from read_csv('/Users/adrienblanquer/Downloads/samedi_dusseldorf.csv')
    """
)

BinderException: Binder Error: Set operations can only apply to expressions with the same number of result columns